In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

import import_ipynb
from notebooks import DataPreProcessing as dp

In [ ]:
# Get preprocessed data from DataPreProcessing notebook
X_train = dp.X_train
X_test = dp.X_test
y_train = dp.y_train
y_test = dp.y_test
clean_text = dp.clean_text

# Available sub-models
MODEL_TYPES = ['DecisionTree', 'KNN', 'SVM']

In [ ]:
# ---------- PATHS ----------
SAVE_DIR = os.path.join(os.getcwd(), 'models')

def _save_path(model_type):
    return os.path.join(SAVE_DIR, f'classic_ml_{model_type.lower()}.pkl')

In [ ]:
# ---------- TRAIN ----------
def train(X_train, y_train, X_test, y_test, model_type='DecisionTree'):
    """Train a classic ML model.
    model_type: 'DecisionTree', 'KNN', or 'SVM'
    """
    # KNN uses a different vectorizer config
    if model_type == 'KNN':
        vectorizer = TfidfVectorizer(max_features=2000, stop_words='english')
    else:
        vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))

    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    if model_type == 'DecisionTree':
        model = DecisionTreeClassifier(max_depth=10, min_samples_split=5, random_state=42)
    elif model_type == 'KNN':
        model = KNeighborsClassifier(n_neighbors=5, metric='cosine')
    elif model_type == 'SVM':
        model = SVC(kernel='linear', C=1.0, random_state=42)
    else:
        raise ValueError(f'Unknown model_type: {model_type}. Use one of {MODEL_TYPES}')

    model.fit(X_train_vec, y_train)

    y_pred = model.predict(X_test_vec)
    acc = accuracy_score(y_test, y_pred)
    print(f'{model_type} Test Accuracy: {acc*100:.2f}%')
    print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

    save_model(model, vectorizer, model_type)
    return model, vectorizer, acc

In [ ]:
# ---------- SAVE / LOAD ----------
def save_model(model, vectorizer, model_type):
    os.makedirs(SAVE_DIR, exist_ok=True)
    path = _save_path(model_type)
    with open(path, 'wb') as f:
        pickle.dump({'model': model, 'vectorizer': vectorizer, 'model_type': model_type}, f)
    print(f'Model saved to {path}')

def load_model(model_type='DecisionTree'):
    path = _save_path(model_type)
    with open(path, 'rb') as f:
        data = pickle.load(f)
    return data['model'], data['vectorizer']

def has_saved_model(model_type='DecisionTree'):
    return os.path.exists(_save_path(model_type))

In [ ]:
# ---------- PREDICT ----------
def predict_message(message, model=None, vectorizer=None, model_type='DecisionTree'):
    """Predict whether a message is Spam or Ham."""
    if model is None or vectorizer is None:
        model, vectorizer = load_model(model_type)
    cleaned = clean_text(message)
    msg_vec = vectorizer.transform([cleaned])
    prediction = model.predict(msg_vec)[0]
    label = 'Spam' if prediction == 1 else 'Ham'

    # SVM uses decision_function for confidence, others use predict_proba
    if isinstance(model, SVC):
        decision_score = model.decision_function(msg_vec)[0]
        confidence = 1 / (1 + np.exp(-abs(decision_score))) * 100
    else:
        probabilities = model.predict_proba(msg_vec)[0]
        confidence = max(probabilities) * 100

    return label, confidence